## Purpose: We want to see if invoice page extraction is applicable on all document types. Therefore fetch all document that we did not recognize, and check theirs invoice page predictions

In [8]:
import pandas as pd
from python_utilities.db_connection import DbConnection

analytics_db = DbConnection("ANALYTICS", "PROD_RDS")

INFO [2026-07-21 08:48:04] - PYTHON_UTILITIES - secret_utilities.py - get_db_secret_config - Credentials for database were read from secret.ini file


In [9]:

# All invoice-page predictions (is_invoice_inside = True) for documents that were
# NOT recognized as ladung / pfub_erlass / drittauskunft / vermogenverzeichnis /
# egvp_standalone_invoice (reason = must_pay).
query_unknown_doc_invoice_inside = """
SELECT *
FROM llm_attachments_predictions lap
WHERE lap.type = "invoice_detection_egvp"
  AND lap.subtype = "is_invoice_inside"
  AND lap.value = "'True'"
  AND lap.created_at >= "2026-06-01"
  AND NOT EXISTS (
        SELECT 1
        FROM llm_attachments_predictions known
        WHERE known.attachment_id = lap.attachment_id
          AND (
                (known.type = "aftercourt_classification_ladung" AND known.subtype = "class_pred" AND known.value = "'True'")
             OR (known.type = "pfub_erlass_egvp"                 AND known.subtype = "is_pfub"    AND known.value = "'True'")
             OR (known.type = "drittauskunft_egvp"               AND known.subtype = "is_dritt"   AND known.value = "'True'")
             OR (known.type = "vermogenverzeichnis_egvp"         AND known.subtype = "is_va"      AND known.value = "'True'")
             OR (known.type = "egvp_standalone_invoice"          AND known.subtype = "reason"     AND known.value = "'must_pay'")
          )
  )
"""


In [10]:
data = analytics_db.sql_to_df(query_unknown_doc_invoice_inside)

In [11]:
data

,id,created_at,model_name,type,subtype,value,attachment_id
0,9887774,2026-06-01 06:28:54,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',66979629
1,9887791,2026-06-01 06:28:55,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',66979661
2,9969998,2026-06-01 22:48:44,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',66990795
3,9970015,2026-06-01 22:48:45,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',66990884
4,9970066,2026-06-01 22:48:49,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',66990893
...,...,...,...,...,...,...,...
3565,13772425,2026-07-21 06:36:54,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',29047238589980-1
3566,13775492,2026-07-21 06:41:57,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',29047219526428-1
3567,13775639,2026-07-21 06:47:17,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',29047239308444-1
3568,13775660,2026-07-21 06:47:18,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',29047250408348-1


In [12]:
data.type.value_counts()

type
invoice_detection_egvp    3570
Name: count, dtype: int64

In [13]:
data.attachment_id.is_unique

True

In [14]:
data.attachment_id.str.contains("-").sum()

np.int64(464)

In [15]:
zendesk_letter = data[data.attachment_id.str.contains("-")]
egvp = data[~data.attachment_id.str.contains("-")]

In [16]:
zendesk_letter

,id,created_at,model_name,type,subtype,value,attachment_id
2647,12570911,2026-07-07 21:42:31,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',28738594423324-1
2653,12572335,2026-07-07 21:44:25,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',28738623160092-1
2654,12572356,2026-07-07 21:44:26,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',28738513000860-1
2655,12572377,2026-07-07 21:44:28,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',28738596332572-1
2656,12572398,2026-07-07 21:44:30,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',28738585072796-1
...,...,...,...,...,...,...,...
3565,13772425,2026-07-21 06:36:54,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',29047238589980-1
3566,13775492,2026-07-21 06:41:57,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',29047219526428-1
3567,13775639,2026-07-21 06:47:17,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',29047239308444-1
3568,13775660,2026-07-21 06:47:18,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',29047250408348-1


In [19]:
zendesk_letter_sample = zendesk_letter.sample(60, random_state=42)
egvp_sample = egvp.sample(60, random_state=42)


zendesk_letter_sample['source'] = 'zendesk_letter'
egvp_sample['source'] = 'egvp'

final_sample = pd.concat([zendesk_letter_sample, egvp_sample], ignore_index=True)

In [20]:
final_sample

,id,created_at,model_name,type,subtype,value,attachment_id,source
0,12579628,2026-07-07 23:47:57,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',28748168015644-1,zendesk_letter
1,12606088,2026-07-08 13:50:11,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',28752685625244-1,zendesk_letter
2,12785308,2026-07-09 16:50:13,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',28791524863516-1,zendesk_letter
3,13690590,2026-07-20 13:03:59,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',29026450613532-1,zendesk_letter
4,13300294,2026-07-15 16:02:11,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',28406495425052-1,zendesk_letter
...,...,...,...,...,...,...,...,...
115,10051534,2026-06-02 17:29:35,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',67130739,egvp
116,12444326,2026-07-07 01:05:33,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',71165836,egvp
117,10085902,2026-06-03 10:40:50,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',67405113,egvp
118,11433637,2026-06-23 17:42:17,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',69429679,egvp


In [ ]:
#final_sample.to_csv("invoice_page_predictions_unknown_docs_sample.csv", index=False)

In [22]:
final_sample

,id,created_at,model_name,type,subtype,value,attachment_id,source
0,12579628,2026-07-07 23:47:57,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',28748168015644-1,zendesk_letter
1,12606088,2026-07-08 13:50:11,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',28752685625244-1,zendesk_letter
2,12785308,2026-07-09 16:50:13,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',28791524863516-1,zendesk_letter
3,13690590,2026-07-20 13:03:59,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',29026450613532-1,zendesk_letter
4,13300294,2026-07-15 16:02:11,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',28406495425052-1,zendesk_letter
...,...,...,...,...,...,...,...,...
115,10051534,2026-06-02 17:29:35,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',67130739,egvp
116,12444326,2026-07-07 01:05:33,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',71165836,egvp
117,10085902,2026-06-03 10:40:50,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',67405113,egvp
118,11433637,2026-06-23 17:42:17,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',69429679,egvp


## Investigate the sample: fetch all predictions per attachment + download the PDFs

For every `attachment_id` in the sampled dataframe we:
1. fetch **all** predictions for that document via `get_data_by_attachment_id`,
2. download the document PDF to `assets/pdfs/tmp/unknown_rejected/`,
3. store the combined predictions to a CSV that the Streamlit review app reads.


In [23]:
import os
import sys

import boto3
import pandas as pd
from tqdm.auto import tqdm

sys.path.append("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation")
from utils.prod_utils import get_data_by_attachment_id

# analytics_db is already created above; only the S3 client is missing.
session = boto3.Session(profile_name="739275445236_DataScienceUser")
s3 = session.client("s3")

PDF_DOWNLOAD_DIR = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/tmp/unknown_rejected"
os.makedirs(PDF_DOWNLOAD_DIR, exist_ok=True)

SAMPLE_CSV = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/classification/invoice_page_detection/invoice_page_predictions_unknown_docs_sample.csv"
sample_df = pd.read_csv(SAMPLE_CSV)
sample_df.head()


/Users/melih.gorgulu/miniconda3/envs/aftercourt-automation/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO [2026-07-21 09:06:44] - Found credentials in shared credentials file: ~/.aws/credentials


,id,created_at,model_name,type,subtype,value,attachment_id,source
0,12579628,2026-07-07 23:47:57,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',28748168015644-1,zendesk_letter
1,12606088,2026-07-08 13:50:11,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',28752685625244-1,zendesk_letter
2,12785308,2026-07-09 16:50:13,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',28791524863516-1,zendesk_letter
3,13690590,2026-07-20 13:03:59,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',29026450613532-1,zendesk_letter
4,13300294,2026-07-15 16:02:11,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',28406495425052-1,zendesk_letter


In [24]:
# Fetch all predictions per attachment and download the corresponding PDF.
source_by_att = dict(zip(sample_df["attachment_id"], sample_df["source"]))

all_preds = []
failed = []

for att_id in tqdm(sample_df["attachment_id"].unique(), desc="attachments"):
    try:
        preds = get_data_by_attachment_id(
            att_id,
            analytics_db,
            s3,
            pdf_download=True,
            pdf_download_dir=PDF_DOWNLOAD_DIR,
            verbose=False,
        )
        if preds is None or preds.empty:
            failed.append(att_id)
            continue
        preds = preds.copy()
        preds["source"] = source_by_att.get(att_id)
        preds["pdf_path"] = os.path.join(PDF_DOWNLOAD_DIR, f"{att_id}.pdf")
        all_preds.append(preds)
    except Exception as e:  # noqa: BLE001
        print(f"Failed for {att_id}: {e}")
        failed.append(att_id)

all_preds_df = pd.concat(all_preds, ignore_index=True)
print(
    f"Collected predictions for {all_preds_df['attachment_id'].nunique()} attachments "
    f"({len(all_preds_df)} rows); failed: {len(failed)}"
)


attachments: 100%|██████████| 120/120 [01:12<00:00,  1.67it/s]

Collected predictions for 120 attachments (2426 rows); failed: 0


In [25]:
# Persist the combined predictions so the Streamlit review app can load them.
ALL_PREDS_CSV = os.path.join(PDF_DOWNLOAD_DIR, "unknown_rejected_all_predictions.csv")
all_preds_df.to_csv(ALL_PREDS_CSV, index=False)
print("Saved:", ALL_PREDS_CSV)
if failed:
    print("Failed attachment_ids:", failed)


Saved: /Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/tmp/unknown_rejected/unknown_rejected_all_predictions.csv


## Review the documents

Launch the Streamlit review app (renders each PDF next to its predictions and lets you save a verdict):

```bash
streamlit run notebooks/classification/invoice_page_detection/review_invoice_pages_unknown_docs_streamlit.py
```

Verdicts are saved to `invoice_page_review_results.csv` in this folder.


## Review results analysis

Summary of the manual review (`invoice_page_review_results.csv`): how often the
`invoice_detection_egvp` invoice-page prediction was judged correct on documents
that were **not** recognized as ladung / pfub_erlass / drittauskunft /
vermögensverzeichnis / standalone-invoice.


In [31]:
REVIEW_CSV = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/classification/invoice_page_detection/invoice_page_review_results.csv"
review = pd.read_csv(REVIEW_CSV)

n_sampled = final_sample["attachment_id"].nunique()
n_reviewed = review["attachment_id"].nunique()
print(f"Reviewed {n_reviewed} / {n_sampled} sampled attachments\n")

print("Verdict distribution:")
print(review["verdict"].value_counts(), "\n")

# Correct rate among decided verdicts (exclude 'unsure' / 'unreviewed')
decided = review[review["verdict"].isin(["correct", "incorrect"])]
if len(decided):
    print(f"Correct rate on decided ({len(decided)}): {(decided['verdict'] == 'correct').mean():.1%}\n")


Reviewed 120 / 120 sampled attachments

Verdict distribution:
verdict
correct      71
incorrect    49
Name: count, dtype: int64 

Correct rate on decided (120): 59.2%



In [32]:
# Attach the parsed (Textract) document text to each reviewed attachment
text_by_att = (
    all_preds_df.assign(attachment_id=all_preds_df["attachment_id"].astype(str))
    .dropna(subset=["text"])
    .groupby("attachment_id")["text"]
    .first()
)
review["text"] = review["attachment_id"].astype(str).map(text_by_att)
print(f"Text attached for {review['text'].notna().sum()} / {len(review)} rows")
review


Text attached for 120 / 120 rows


,attachment_id,source,verdict,notes,reviewed_at,text
0,28748168015644-1,zendesk_letter,correct,NaN,2026-07-21T07:27:29,Landesamt für\nBERLIN\nBürger- und Ordnungsang...
1,28752685625244-1,zendesk_letter,correct,NaN,2026-07-21T07:28:39,Stadt Zürich\nStadt Zürich\nBevölkerungsamt\nB...
2,28791524863516-1,zendesk_letter,incorrect,NaN,2026-07-21T07:29:14,Amtsgericht Coburg\nMahnsache ON Energie Deuts...
3,29026450613532-1,zendesk_letter,correct,NaN,2026-07-21T07:29:27,Amtsgericht Moers\n-Geschäftsstelle-\n-502- Am...
4,28406495425052-1,zendesk_letter,correct,NaN,2026-07-21T07:29:37,Kontakt und Information\nBaden-Württemberg\nBü...
...,...,...,...,...,...,...
115,71165836,egvp,incorrect,NaN,2026-07-21T08:15:00,Amtsgericht Coburg\nMahnsac Octopus Energy Ger...
116,67405113,egvp,correct,NaN,2026-07-21T08:15:04,Soestfeld 15\n59514 Welver\nMobil 0178/1101582...
117,69429679,egvp,incorrect,NaN,2026-07-21T08:15:15,Amtsgericht Hünfeld\nMahnsache Deutsche Parkau...
118,72548635,egvp,incorrect,NaN,2026-07-21T08:15:29,Amtsgericht Wedding\nMahnsache MILES Mobility ...


In [33]:
# Verdict breakdown by source, with correct rate per source
by_source = review.groupby("source")["verdict"].value_counts().unstack(fill_value=0)
for col in ["correct", "incorrect", "unsure", "unreviewed"]:
    if col not in by_source.columns:
        by_source[col] = 0

decided_cnt = by_source["correct"] + by_source["incorrect"]
by_source["correct_rate"] = (by_source["correct"] / decided_cnt.replace(0, pd.NA)).round(3)
by_source


verdict,correct,incorrect,unsure,unreviewed,correct_rate
source,,,,,
egvp,27,33,0,0,0.450
zendesk_letter,44,16,0,0,0.733


In [34]:
# Cases to follow up on (incorrect / unsure)
review[review["verdict"].isin(["incorrect", "unsure"])][
    ["attachment_id", "source", "verdict", "notes"]
].reset_index(drop=True)


,attachment_id,source,verdict,notes
0,28791524863516-1,zendesk_letter,incorrect,NaN
1,29013377008412-1,zendesk_letter,incorrect,NaN
2,28816610383644-1,zendesk_letter,incorrect,NaN
3,29013592839964-1,zendesk_letter,incorrect,NaN
4,28816563010332-1,zendesk_letter,incorrect,NaN
5,28738594423324-1,zendesk_letter,incorrect,NaN
6,28878994610460-1,zendesk_letter,incorrect,NaN
7,28822031625628-1,zendesk_letter,incorrect,NaN
8,28804978585116-1,zendesk_letter,incorrect,NaN
9,28822049201308-1,zendesk_letter,incorrect,NaN


In [35]:
review

,attachment_id,source,verdict,notes,reviewed_at,text
0,28748168015644-1,zendesk_letter,correct,NaN,2026-07-21T07:27:29,Landesamt für\nBERLIN\nBürger- und Ordnungsang...
1,28752685625244-1,zendesk_letter,correct,NaN,2026-07-21T07:28:39,Stadt Zürich\nStadt Zürich\nBevölkerungsamt\nB...
2,28791524863516-1,zendesk_letter,incorrect,NaN,2026-07-21T07:29:14,Amtsgericht Coburg\nMahnsache ON Energie Deuts...
3,29026450613532-1,zendesk_letter,correct,NaN,2026-07-21T07:29:27,Amtsgericht Moers\n-Geschäftsstelle-\n-502- Am...
4,28406495425052-1,zendesk_letter,correct,NaN,2026-07-21T07:29:37,Kontakt und Information\nBaden-Württemberg\nBü...
...,...,...,...,...,...,...
115,71165836,egvp,incorrect,NaN,2026-07-21T08:15:00,Amtsgericht Coburg\nMahnsac Octopus Energy Ger...
116,67405113,egvp,correct,NaN,2026-07-21T08:15:04,Soestfeld 15\n59514 Welver\nMobil 0178/1101582...
117,69429679,egvp,incorrect,NaN,2026-07-21T08:15:15,Amtsgericht Hünfeld\nMahnsache Deutsche Parkau...
118,72548635,egvp,incorrect,NaN,2026-07-21T08:15:29,Amtsgericht Wedding\nMahnsache MILES Mobility ...


In [38]:
review['widerspruch'] = review['text'].str.contains('widerspruch', case=False, na=False)

In [40]:
review['widerspruch'].value_counts()

widerspruch
False    70
True     50
Name: count, dtype: int64

In [41]:
incorrect = review[review['verdict'] == 'incorrect']
incorrect['widerspruch'].value_counts()

widerspruch
True     48
False     1
Name: count, dtype: int64

In [42]:
# widerspruch docs are classified as invoice wrongly, this can be fixed with regex. If its fixed:
review['verdict_fixed'] = review.apply(lambda row: 'correct' if row['verdict'] == 'incorrect' and row['widerspruch'] else row['verdict'], axis=1)

In [44]:
review['verdict_fixed'].value_counts()

verdict_fixed
correct      119
incorrect      1
Name: count, dtype: int64

In [45]:
# Verdict breakdown by source, with correct rate per source
by_source = review.groupby("source")["verdict_fixed"].value_counts().unstack(fill_value=0)
for col in ["correct", "incorrect", "unsure", "unreviewed"]:
    if col not in by_source.columns:
        by_source[col] = 0

decided_cnt = by_source["correct"] + by_source["incorrect"]
by_source["correct_rate"] = (by_source["correct"] / decided_cnt.replace(0, pd.NA)).round(3)
by_source


verdict_fixed,correct,incorrect,unsure,unreviewed,correct_rate
source,,,,,
egvp,59,1,0,0,0.983
zendesk_letter,60,0,0,0,1.000
